# Unit 1, Lecture 2: Four architectures, one task

The tools are identical. The loop is identical. **Only the architecture changes.**

That is the whole experiment. If the loop varied too, you could not tell whether a
difference in behaviour came from the architecture or from the plumbing.

| Rung | Architecture | Adds |
|---|---|---|
| 1 | Simple reflex | nothing, this is the floor |
| 2 | Model based reflex | internal state |
| 3 | Goal based | foresight |
| 4 | Utility based | preference |

There are three hotels on file, and the Taj Palace is **full** on 14 August. That is
deliberate. Watch which architectures recover from it and which do not.

## Setup

In [1]:
from cse476.lanes import get_client, MODEL, describe
from cse476.architectures import (
    HOTELS, ROOMS,
    reflex, ModelBasedAgent, goal_based, utility_based,
)

print(describe())
client = get_client()

for name, d in HOTELS.items():
    free = ROOMS.get((name, "2026-08-14"))
    print(f"{name:14} Rs {d['rate']:>6}  {d['km_from_campus']:>5} km  "
          f"rating {d['rating']}  rooms free: {free}")

Lane: Microsoft Foundry (foundry, billed)  |  Model: chat-demo
Taj Palace     Rs  14500    2.1 km  rating 4.8  rooms free: 0
Radisson Blu   Rs   6200    8.4 km  rating 4.2  rooms free: 11
Hotel Meera    Rs   2800   14.9 km  rating 3.4  rooms free: 6


## Rung 1, simple reflex

One percept in, one action out, no memory.

Notice how the architecture is enforced: not by trusting the prompt, but by
`max_steps=2`, which makes a second round structurally impossible. **A prompt is a
request. A budget is a guarantee.** When the two disagree, only one of them is load
bearing.

In [2]:
print(reflex(client, MODEL, "Is there a room at Taj Palace on 2026-08-14?"))

  reflex [1] get_room_availability({'hotel': 'Taj Palace', 'date': '2026-08-14'}) -> Taj Palace on 2026-08-14: 0 rooms available.
No — there are 0 rooms available at Taj Palace on 2026-08-14.


Correct, and useless. It told you the obvious first choice is full and stopped.
It has no way to try anything else, because trying something else requires knowing
you already tried the first thing, and there is no *it* that remembers.

## Rung 2, model based reflex

Same reflex behaviour, plus internal state.

The only difference in the code is that `messages` moved from a local variable into
an instance attribute. That is the entire distance between rung one and rung two.

In [3]:
agent = ModelBasedAgent(client, MODEL)

print(agent.ask("Is there a room at Taj Palace on 2026-08-14?"))
print()
print(agent.ask("What was the hotel I just asked about?"))

  model  [1] get_room_availability({'hotel': 'Taj Palace', 'date': '2026-08-14'}) -> Taj Palace on 2026-08-14: 0 rooms available.
No — there are 0 rooms available at Taj Palace on 2026-08-14.

You asked about Taj Palace.


In [4]:
# The state is a plain list. Look at it.
for m in agent.messages:
    role = m["role"] if isinstance(m, dict) else m.role
    content = (m.get("content") if isinstance(m, dict) else m.content) or ""
    print(f"{role:10} {str(content)[:80]}")

system     You are a lookup service with a memory of this conversation. Answer using tools.
user       Is there a room at Taj Palace on 2026-08-14?
assistant  
tool       Taj Palace on 2026-08-14: 0 rooms available.
assistant  No — there are 0 rooms available at Taj Palace on 2026-08-14.
user       What was the hotel I just asked about?
assistant  You asked about Taj Palace.


It remembers. It still does not **aim**. There is no notion of a desired end
state, so it cannot work out a route to one. Better conversation, same dead end.

## Rung 3, goal based

Now we stop dictating the shape of the answer. Same three tools, same loop, budget
raised from 2 to 8, and a prompt that gives an outcome instead of an instruction.

Watch the trace. It finds the Taj Palace full and moves on **by itself**. Nothing in
your code says "if full, try another".

In [5]:
answer = goal_based(
    client, MODEL,
    "Find me a hotel with a room available on 2026-08-14."
)
print()
print(answer)

  goal   [1] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  goal   [2] get_room_availability({'hotel': 'Taj Palace', 'date': '2026-08-14'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  goal   [2] get_room_availability({'hotel': 'Radisson Blu', 'date': '2026-08-14'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  goal   [2] get_room_availability({'hotel': 'Hotel Meera', 'date': '2026-08-14'}) -> Hotel Meera on 2026-08-14: 6 rooms available.
  goal   [3] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.
  goal   [3] get_hotel_details({'hotel': 'Hotel Meera'}) -> Hotel Meera: Rs 2800 per night, 14.9 km from campus, guest rating 3.4 out of 5.

I found two hotels with rooms available on 2026-08-14:

- Radisson Blu — 11 rooms available. Rate: Rs 6,200/night. Distance: 8.4 km from campus. Guest rating: 4.2/5.  
- Hotel Meera — 6 rooms available. Rate: Rs 2,800/night. Distance: 14

## Rung 4, utility based

A goal tells you when to stop. It does not tell you which of three acceptable
answers is best.

The weights below **are** the utility function. Written down, passed in, and
therefore arguable.

In [6]:
answer = utility_based(
    client, MODEL,
    goal="Find me a hotel with a room available on 2026-08-14.",
    preferences={"price": 0.5, "distance from campus": 0.3, "rating": 0.2},
)
print()
print(answer)

  utility[1] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  utility[2] get_room_availability({'hotel': 'Taj Palace', 'date': '2026-08-14'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  utility[2] get_room_availability({'hotel': 'Radisson Blu', 'date': '2026-08-14'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  utility[2] get_room_availability({'hotel': 'Hotel Meera', 'date': '2026-08-14'}) -> Hotel Meera on 2026-08-14: 6 rooms available.
  utility[3] get_hotel_details({'hotel': 'Taj Palace'}) -> Taj Palace: Rs 14500 per night, 2.1 km from campus, guest rating 4.8 out of 5.
  utility[4] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.
  utility[5] get_hotel_details({'hotel': 'Hotel Meera'}) -> Hotel Meera: Rs 2800 per night, 14.9 km from campus, guest rating 3.4 out of 5.

Summary of what I checked
- Date requested: 2026-08-14
- Hotels on file: Taj Palace, Radisson Blu, 

### Now change the weights

Same code. Same tools. Different preference, and the recommendation should move.

If it does not move, your weights are not doing anything, and the honest thing is to
say so rather than hide it. A utility function that never changes the answer is
decoration.

In [7]:
answer = utility_based(
    client, MODEL,
    goal="Find me a hotel with a room available on 2026-08-14.",
    preferences={"distance from campus": 0.7, "price": 0.2, "rating": 0.1},
)
print()
print(answer)

  utility[1] list_hotels() -> Hotels on file: Taj Palace, Radisson Blu, Hotel Meera
  utility[2] get_room_availability({'hotel': 'Taj Palace', 'date': '2026-08-14'}) -> Taj Palace on 2026-08-14: 0 rooms available.
  utility[2] get_room_availability({'hotel': 'Radisson Blu', 'date': '2026-08-14'}) -> Radisson Blu on 2026-08-14: 11 rooms available.
  utility[2] get_room_availability({'hotel': 'Hotel Meera', 'date': '2026-08-14'}) -> Hotel Meera on 2026-08-14: 6 rooms available.
  utility[2] get_hotel_details({'hotel': 'Taj Palace'}) -> Taj Palace: Rs 14500 per night, 2.1 km from campus, guest rating 4.8 out of 5.
  utility[2] get_hotel_details({'hotel': 'Radisson Blu'}) -> Radisson Blu: Rs 6200 per night, 8.4 km from campus, guest rating 4.2 out of 5.
  utility[2] get_hotel_details({'hotel': 'Hotel Meera'}) -> Hotel Meera: Rs 2800 per night, 14.9 km from campus, guest rating 3.4 out of 5.

Thanks — I checked availability for 2026-08-14 and scored every hotel that has at least one room fr

## What the four traces show

| Architecture | Calls | Outcome |
|---|---|---|
| Reflex | 1 | Taj Palace is full. Correct, useless. |
| Model based | 2 to 4 | Same dead end, but remembers the conversation. |
| Goal based | 4 to 6 | Recovers on its own. First one that solves the task. |
| Utility based | 6 to 9 | Solves it **and** justifies it. |

Step counts vary between runs, because the model is choosing. That variability is
the point, and it is also why testing agents is hard, which is Unit 5.

**The cost the table also shows:** going up the ladder multiplied the API calls by
roughly six. Every rung is bought with money and latency. Choosing rung four when
rung two would do is not sophistication, it is waste.

## Your turn

**1. The weight experiment.** Run `utility_based` three times with clearly different
weightings. Record the recommendation each time. If all three agree, say so and
explain why the weights failed to bite.

**2. Break rung 3 on purpose.** Ask the goal based agent for something no hotel
satisfies, for example a room under Rs 500. Watch it exhaust the budget. Then argue,
in one sentence, whether `max_steps=8` is the right number for this task.

**3. PEAS.** Write the PEAS description for a system you use daily. Four lines. Then
say which rung of the ladder it needs, and why the environment forces that.

In [ ]:
# your work here
